# 3. Fixing Skewness and Building the Final Dataset
## Project: Global Life Expectancy Paradox

**Purpose:** identify heavily right-skewed variables, apply log transformations where appropriate, create life-expectancy features, encode regions, and export the final feature-engineered CSV.

**Why skewness matters:** population, population density, GDP per capita, homicide rate and CO₂ emissions can have a small number of very large values. A `log1p(x)` transformation computes `log(1+x)`, reducing the influence of extreme magnitudes while allowing zeros. It is not a guarantee of normality, so skewness is checked again after transformation.

**Important:** transformed columns are added alongside original columns; source values are not overwritten.

In [ ]:
# In Google Colab, upload CLEANED_DATASET_FINAL(1).csv when prompted.
from google.colab import files
uploaded = files.upload()
import io, pandas as pd, numpy as np
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Dataset shape:", df.shape)
display(df.head())

## Step 1 — Create target/context features
The dataset has male and female life expectancy rather than one overall field. We derive:
- `avg_life_expectancy`: arithmetic mean of male and female life expectancy.
- `female_male_life_expectancy_gap`: female minus male life expectancy, in years.

In [ ]:
df['avg_life_expectancy'] = (df['life_expectancy_male'] + df['life_expectancy_female']) / 2
df['female_male_life_expectancy_gap'] = df['life_expectancy_female'] - df['life_expectancy_male']
print("Average life expectancy summary:")
display(df['avg_life_expectancy'].describe())
print("Mean female-minus-male gap:", round(df['female_male_life_expectancy_gap'].mean(), 2), "years")

## Step 2 — Measure skewness before transformation
Skewness near 0 indicates approximate symmetry; a large positive value indicates a long right tail. The threshold below is a descriptive rule of thumb, not a strict statistical test.

In [ ]:
numeric = df.select_dtypes(include=np.number)
skew_before = numeric.skew().sort_values(key=lambda s: s.abs(), ascending=False)
display(skew_before.rename('skewness_before').to_frame().head(15))

## Step 3 — Apply log1p to selected right-skewed features
Only non-negative features are selected for this log transformation. `log1p` handles zeros safely. GDP growth is not selected because it can be negative.

In [ ]:
skew_candidates = ['population', 'pop_density', 'gdp_per_capita', 'homicide_rate', 'co2_emissions']
for col in skew_candidates:
    if col in df.columns:
        df[col + '_log1p'] = np.log1p(df[col].clip(lower=0))
after = pd.DataFrame({
    'before': [df[c].skew() for c in skew_candidates],
    'after_log1p': [df[c + '_log1p'].skew() for c in skew_candidates]
}, index=skew_candidates)
display(after)

## Step 4 — Scale numeric features and encode region
This creates z-score and min-max versions of numeric columns and one-hot indicators for region. The original measurements remain available. The final CSV is intended for exploration and coursework; for model evaluation, fit preprocessing objects on training data only.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    df[col + '_scaled'] = StandardScaler().fit_transform(df[[col]]).ravel()
for col in numeric_cols:
    df[col + '_minmax'] = MinMaxScaler().fit_transform(df[[col]]).ravel()
region_dummies = pd.get_dummies(df['region'], prefix='region', dtype=int)
final_df = pd.concat([df, region_dummies], axis=1)
print("Final shape:", final_df.shape)
print("Missing cells:", int(final_df.isna().sum().sum()))
display(final_df.head())

## Step 5 — Explore the life-expectancy paradox
These are associations in a country-level snapshot, not proof of cause and effect. Correlations can be affected by confounders and data-year differences. Use this as an exploratory analysis, not a causal conclusion.

In [ ]:
correlations = final_df.select_dtypes(include=np.number).corr()['avg_life_expectancy'].sort_values(ascending=False)
display(correlations.drop(labels=['avg_life_expectancy'], errors='ignore').head(10).rename('correlation_with_average_life_expectancy').to_frame())
display(correlations.drop(labels=['avg_life_expectancy'], errors='ignore').tail(10).rename('correlation_with_average_life_expectancy').to_frame())
region_means = final_df.groupby('region')['avg_life_expectancy'].agg(['count','mean']).sort_values('mean')
display(region_means)

## Step 6 — Export final feature-engineered dataset

In [ ]:
output_name = 'global_life_expectancy_feature_engineered.csv'
final_df.to_csv(output_name, index=False)
from google.colab import files
files.download(output_name)